## Build the Silver Layer Transformation for Orders

In [0]:


from pyspark.sql.functions import to_timestamp
from pyspark.sql.functions import col, month ,year, dayofweek

olist_orders_df = spark.read.table("upskill.pyspark_learning.bronze_olist_orders")

# Cast Timestamps: Convert all 5 date/timestamp string columns to TimestampType()
timestamp_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date","order_delivered_customer_date","order_estimated_delivery_date"]

for c in timestamp_cols:
    olist_orders_df = olist_orders_df.withColumn(c, to_timestamp(c, "yyyy-MM-dd HH:mm:ss"))


# Filter Bad Data: Filter out invalid orders (keep only records where order_id is NOT NULL and customer_id is NOT NULL).
olist_orders_df = olist_orders_df.filter(col("order_id").isNotNull() & col("customer_id").isNotNull())
   
#Derived Date Columns: Extract order_purchase_year, order_purchase_month, and order_purchase_day_of_week (e.g., Mon/Tue or 1-7) from order_purchase_timestamp.
olist_orders_df = olist_orders_df.withColumn("order_purchase_year",year("order_purchase_timestamp"))
olist_orders_df = olist_orders_df.withColumn("order_purchase_month",month("order_purchase_timestamp"))
olist_orders_df = olist_orders_df.withColumn("order_purchase_day_of_week",dayofweek("order_purchase_timestamp"))

# Save to Delta: Save the transformed DataFrame as silver_orders partitioned by order_purchase_year
olist_orders_df.write\
    .format("delta")\
    .mode("overwrite")\
    .partitionBy("order_purchase_year")\
    .saveAsTable("upskill.pyspark_learning.silver_olist_orders")   

## Build the Silver Layer Transformation for Orders Payment

In [0]:
from pyspark.sql.functions import initcap,regexp_replace
from pyspark.sql.functions import col

olist_orderPayment_df = spark.read.table("upskill.pyspark_learning.bronze_olist_orderPayment")

#Keep records where order_id is NOT NULL and payment_value > 0.
olist_orderPayment_df = olist_orderPayment_df.filter(col("order_id").isNotNull() & (col("payment_value") > 0))

#payment_type to replace underscores _ with spaces and convert to title case (e.g., credit_card $\rightarrow$ Credit Card)
olist_orderPayment_df = olist_orderPayment_df.withColumn(
    "payment_type", 
    initcap(regexp_replace(col("payment_type"), "_", " "))
)

# 4. Save to Silver Delta Table
olist_orderPayment_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("upskill.pyspark_learning.silver_olist_orderPayment")


# Quick Verification
display(spark.sql("SELECT * FROM upskill.pyspark_learning.silver_olist_orderPayment LIMIT 5"))

## Build the Silver Layer Transformation for customer

In [0]:
from pyspark.sql.functions import initcap,upper,col

#Read upskill.pyspark_learning.bronze_olist_customer table
olist_customer_df = spark.read.table("upskill.pyspark_learning.bronze_olist_customer")

#Filter out rows where customer_id or customer_unique_id is NULL.
olist_customer_df = olist_customer_df.filter(col('customer_id').isNotNull() & col('customer_unique_id').isNotNull())

#Convert customer_city to Title Case (initcap(col("customer_city"))).
olist_customer_df = olist_customer_df.withColumn('customer_city',initcap(col('customer_city')))

#Convert customer_state to Uppercase (upper(col("customer_state"))).
olist_customer_df = olist_customer_df.withColumn('customer_state',upper(col('customer_state')))

#Save as upskill.pyspark_learning.silver_customers.
olist_customer_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("upskill.pyspark_learning.silver_olist_customer")


# Quick Verification
display(spark.sql("SELECT * FROM upskill.pyspark_learning.silver_olist_customer LIMIT 2"))

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,ingested_at
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,Franca,SP,2026-09-07T07:15:16.939Z
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,Sao Bernardo Do Campo,SP,2026-09-07T07:15:16.939Z


## Build the Silver Layer product table using above product and product_category

In [0]:
from pyspark.sql.functions import initcap , regexp_replace

# Read product category translation table
olist_product_category_df = spark.read.table("upskill.pyspark_learning.bronze_olist_product_category")

# Read products table
olist_products_df = spark.read.table("upskill.pyspark_learning.bronze_olist_products")

# Combine products and category names based on product_category_name
df_products_category_joined = olist_products_df.join(olist_product_category_df, "product_category_name", "left") 

# If product_category_name is null, then set product_category_name_english to 'Unknown' (Handle Missing Translations)
df_products_category_joined = df_products_category_joined.fillna('Unknown', subset=["product_category_name_english"])

#Convert the category name to Title Case using initcap().
df_products_category_joined = df_products_category_joined.withColumn("product_category", 
    initcap(regexp_replace(col("product_category_name_english"), "_", " ")))

#Rename or select columns to keep the dataset tidy (e.g., product_id, product_category as the main category column, product_weight_g..
silver_products_df = df_products_category_joined.select(
    col("product_id"),
    col("product_category"),
    col("product_weight_g"),
    col("product_length_cm"),
    col("product_height_cm"),
    col("product_width_cm")
)

# Create silver product table
silver_products_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("upskill.pyspark_learning.silver_olist_products")

## Build the Silver Layer Transformation for order items dataset

In [0]:
# Read from Bronze Table
silver_order_items_df = spark.read.table("upskill.pyspark_learning.bronze_olist_order_items")

# Filter Bad Data (Valid order_id, product_id, and price > 0)
silver_order_items_df = silver_order_items_df.filter(
    col("order_id").isNotNull() & 
    col("product_id").isNotNull() & 
    (col("price") > 0)
)

# Cast Date String to TimestampType
silver_order_items_df = silver_order_items_df.withColumn(
    "shipping_limit_date", 
    to_timestamp(col("shipping_limit_date"))
)

# Select Cleaned Columns
silver_order_items_df = silver_order_items_df.select(
    col("order_id"),
    col("order_item_id"),
    col("product_id"),
    col("seller_id"),
    col("shipping_limit_date"),
    col("price"),
    col("freight_value")
)

# Save to Silver Delta Table
silver_order_items_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("upskill.pyspark_learning.silver_olist_order_items")


# Quick Verification
display(spark.sql("SELECT * FROM upskill.pyspark_learning.silver_olist_order_items LIMIT 5"))